In [ ]:
import pickle
import cv2
import mediapipe as mp
from collections import deque
import time
import numpy as np
from tensorflow.keras.models import load_model

# 🔹 Load models
static_model = pickle.load(open('Static-Model-Sentence/model.p', 'rb'))['model']
motion_model = load_model('Motion-LSTM-Model/motion_model.h5')

motion_data = pickle.load(open('Motion-LSTM-Model/motion_data.pickle', 'rb'))
actions = motion_data['actions']

# 🔹 MediaPipe
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    min_detection_confidence=0.5,
    max_num_hands=1
)

# 🔹 Feature extraction
def extract_hand_features(hand_landmarks):
    x_, y_ = [], []
    for lm in hand_landmarks.landmark:
        x_.append(lm.x)
        y_.append(lm.y)

    data_aux = []
    for lm in hand_landmarks.landmark:
        data_aux.append(lm.x - min(x_))
        data_aux.append(lm.y - min(y_))

    return data_aux


# 🔥 Text wrapping function
def draw_multiline_text(frame, text, x, y, max_width, line_height=35):
    words = text.split(' ')
    lines = []
    current_line = ""

    for word in words:
        test_line = current_line + word + " "
        text_size = cv2.getTextSize(test_line, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]

        if text_size[0] > max_width:
            lines.append(current_line)
            current_line = word + " "
        else:
            current_line = test_line

    lines.append(current_line)

    for i, line in enumerate(lines):
        cv2.putText(frame, line.strip(), (x, y + i * line_height),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)


# 🔹 Sentence system
buffer = deque(maxlen=10)
sentence = ""
prev_char = ""
last_added_time = 0
cooldown = 1.0

# 🔹 Motion system
sequence = []

# 🔹 Mode
mode = "static"

cap = cv2.VideoCapture(0)

# 🔹 Fullscreen
cv2.namedWindow('Hybrid ISL System', cv2.WND_PROP_FULLSCREEN)
cv2.setWindowProperty('Hybrid ISL System', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)

    # 🔹 Create UI panel
    h, w, _ = frame.shape
    panel_width = 500

    panel = np.zeros((h, panel_width, 3), dtype=np.uint8)

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    display_char = ""
    current_time = time.time()

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        features = extract_hand_features(hand_landmarks)

        mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        # 🔥 MOTION MODE
        if mode == "motion":
            sequence.append(features)
            sequence = sequence[-30:]

            if len(sequence) == 30:
                pred = motion_model.predict(np.expand_dims(sequence, axis=0))[0]
                action = actions[np.argmax(pred)]
                display_char = action

                if (current_time - last_added_time) > cooldown:
                    sentence += action + " "
                    last_added_time = current_time
                    prev_char = ""

        # 🔥 STATIC MODE
        else:
            prediction = static_model.predict([features])[0]
            buffer.append(prediction)

            if len(buffer) == buffer.maxlen:
                stable_char = max(set(buffer), key=buffer.count)
                display_char = stable_char

                if stable_char != prev_char and (current_time - last_added_time) > cooldown:
                    if stable_char == "space":
                        sentence += " "
                    else:
                        sentence += stable_char

                    prev_char = stable_char
                    last_added_time = current_time

    # 🔹 Limit sentence length
    if len(sentence) > 120:
        sentence = sentence[-120:]

    # 🔹 UI Panel Text
    cv2.putText(panel, f'Mode: {mode.upper()}',
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2)

    cv2.putText(panel, f'Current:',
                (20, 100), cv2.FONT_HERSHEY_SIMPLEX,
                0.9, (0, 255, 255), 2)

    cv2.putText(panel, display_char,
                (20, 140), cv2.FONT_HERSHEY_SIMPLEX,
                1.2, (255, 255, 255), 3)

    draw_multiline_text(panel, f'Sentence: {sentence}', 20, 220, 450)

    draw_multiline_text(
        panel,
        'M=Motion | N=Static | Q=Quit | C=Clear | B=Backspace | W=Delete Word',
        20, h - 120, 450, 30
    )

    # 🔹 Merge camera + panel
    combined = np.hstack((frame, panel))

    cv2.imshow('Hybrid ISL System', combined)

    key = cv2.waitKey(1) & 0xFF

    # 🔹 Mode switching
    if key == ord('m'):
        mode = "motion"
        sequence = []

    elif key == ord('n'):
        mode = "static"
        buffer.clear()

    # 🔹 Controls
    elif key == ord('q'):
        break

    elif key == ord('c'):
        sentence = ""
        prev_char = ""

    elif key == ord('b'):
        sentence = sentence[:-1]

    elif key == ord('w'):
        sentence = sentence.rstrip()
        sentence = " ".join(sentence.split(" ")[:-1])

cap.release()
cv2.destroyAllWindows()

1/1 [==============================] - ETA: 0s

In [ ]:
static_model = pickle.load(open('Static-Model-Sentence/model.p', 'rb'))['model']
motion_model = load_model('Motion-LSTM-Model/motion_model.h5')

motion_data = pickle.load(open('Motion-LSTM-Model/motion_data.pickle', 'rb'))